In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/equipment/raw"

for name in ["train", "test"]:
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")
        .load(f"{BASE}/{name}")
        .select("value",
                F.col("_metadata.file_path").alias("_source_file"),
                F.current_timestamp().alias("_ingested_at"))
        .writeStream
        .option("checkpointLocation", f"{BASE}/_checkpoints/{name}")
        .trigger(availableNow=True)
        .toTable(f"workspace.equipment.bronze_{name}")
        .awaitTermination())
    print(name, spark.table(f"workspace.equipment.bronze_{name}").count())